# Gemma 3 12B — Extract Text Decoder + Export Vision ONNX for TRT-LLM

**Purpose**: Prepare the merged VLM model for local TRT-LLM INT4 engine build.

**Input**: Merged VLM on Google Drive at `/content/drive/MyDrive/gemma3-12b-legal-merged-16bit`

**Outputs**:
- Text-only `Gemma3ForCausalLM` (~21 GB) — for TRT-LLM `convert_checkpoint.py` + `trtllm-build`
- SigLIP vision encoder ONNX (~1.5 GB) — for `trtexec` engine build
- Projector ONNX (~50 MB) — for `trtexec` engine build

**Why not quantize here?** TRT-LLM `convert_checkpoint.py` and `trtllm-build` must run in the
same TRT-LLM version. Our Docker container (`Dockerfile.trtllm`, v0.21.0) is the controlled
environment. Doing conversion here risks version mismatch.

**Hardware**: A100 (40GB) recommended. T4 works but slower.

**Time**: ~15-20 minutes total

---

## Architecture (HuggingFace Gemma3 VLM)
```
Gemma3ForConditionalGeneration
  ├── .model (Gemma3Model)
  │     ├── .vision_tower (SiglipVisionModel)            → Export ONNX
  │     ├── .multi_modal_projector (Gemma3MultiModalProjector) → Export ONNX
  │     └── .language_model (Gemma3TextModel)             → Extract as CausalLM
  └── .lm_head (nn.Linear)                               → Include with text decoder
```

**Key**: Sub-models live on `.model`, NOT directly on the top-level class.
The `language_model` is `Gemma3TextModel` (no lm_head), not `Gemma3ForCausalLM`.
We extract via safetensors key remapping to build a proper standalone CausalLM.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers>=4.45.0 safetensors torch onnx accelerate

import torch
import sys
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram_gb:.1f} GB")

import psutil
ram_gb = psutil.virtual_memory().total / 1024**3
print(f"System RAM: {ram_gb:.1f} GB")

## 2. Mount Drive + Find Merged Model

In [ ]:
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Check known paths for the merged model
MERGED_CANDIDATES = [
    '/content/drive/MyDrive/gemma3-12b-legal-merged-16bit',
    '/content/drive/MyDrive/COLAB_PACKAGE/gemma3-12b-legal-merged-16bit',
    '/content/gemma3-12b-legal-merged-16bit',
]

MERGED_DIR = None
for candidate in MERGED_CANDIDATES:
    p = Path(candidate)
    if p.exists() and (p / 'config.json').exists():
        MERGED_DIR = str(p)
        break

if not MERGED_DIR:
    raise FileNotFoundError(
        "Merged model not found! Expected at:\n"
        + "\n".join(f"  {c}" for c in MERGED_CANDIDATES)
        + "\nRun Gemma3_12B_Merge_and_TRT_Export.ipynb first."
    )

print(f"Merged model: {MERGED_DIR}")

# List contents
total_gb = 0
for f in sorted(Path(MERGED_DIR).iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / (1024**2)
        total_gb += size_mb / 1024
        print(f"  {f.name:45s} {size_mb:>8.1f} MB")
print(f"{'':45s} {'─'*12}")
print(f"  {'Total':45s} {total_gb:>7.1f} GB")

## 3. Analyze VLM Config

Verify this is a VLM (`Gemma3ForConditionalGeneration`) and inspect the text/vision split.

In [ ]:
import json

with open(f'{MERGED_DIR}/config.json') as f:
    vlm_config = json.load(f)

arch = vlm_config.get('architectures', ['unknown'])[0]
print(f"Architecture: {arch}")

if 'Conditional' not in arch:
    print("\nThis is already a text-only model (Gemma3ForCausalLM).")
    print("No extraction needed — skip to Cell 7 (Save to Drive).")
    IS_VLM = False
else:
    IS_VLM = True
    print("\nVLM detected — will extract text decoder + export vision components.")

    # Text config
    text_cfg = vlm_config.get('text_config', {})
    print(f"\nText Decoder:")
    print(f"  hidden_size:    {text_cfg.get('hidden_size', vlm_config.get('hidden_size', '?'))}")
    print(f"  num_layers:     {text_cfg.get('num_hidden_layers', vlm_config.get('num_hidden_layers', '?'))}")
    print(f"  num_heads:      {text_cfg.get('num_attention_heads', vlm_config.get('num_attention_heads', '?'))}")
    print(f"  vocab_size:     {text_cfg.get('vocab_size', vlm_config.get('vocab_size', '?'))}")

    # Vision config
    vis_cfg = vlm_config.get('vision_config', {})
    if vis_cfg:
        print(f"\nVision Encoder:")
        print(f"  model_type:     {vis_cfg.get('model_type', '?')}")
        print(f"  hidden_size:    {vis_cfg.get('hidden_size', '?')}")
        print(f"  image_size:     {vis_cfg.get('image_size', '?')}")
        print(f"  patch_size:     {vis_cfg.get('patch_size', '?')}")
        num_patches = (vis_cfg.get('image_size', 384) // vis_cfg.get('patch_size', 14)) ** 2
        print(f"  num_patches:    {num_patches}")
    else:
        print("\nNo vision_config found in config.json")

print(f"\nFull config keys: {list(vlm_config.keys())}")

## 4. Extract Text-Only Model (Safetensors Key Remapping)

The VLM's sub-models live at `model.model.*`, and `language_model` is `Gemma3TextModel`
(no lm_head), NOT `Gemma3ForCausalLM`. We can't just call `.save_pretrained()` on it.

Instead, we remap safetensors keys directly:
- `model.language_model.X` → `model.X` (text decoder weights)
- `lm_head.weight` → `lm_head.weight` (keep as-is)
- Skip `model.vision_tower.*` and `model.multi_modal_projector.*`
- Build a proper `Gemma3ForCausalLM` config from `text_config`

This produces a clean HF checkpoint that TRT-LLM's `convert_checkpoint.py` accepts.

In [ ]:
import os
import json
import math
import torch
from pathlib import Path
from safetensors.torch import load_file, save_file
from collections import OrderedDict

TEXT_ONLY_DIR = '/content/gemma3-12b-legal-text-only'
ONNX_DIR = '/content/trt_onnx_exports'
os.makedirs(TEXT_ONLY_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)

if IS_VLM:
    # ── Step 1: Load all safetensors shards ──
    print("Loading safetensors shards...")
    all_tensors = {}
    shard_files = sorted(Path(MERGED_DIR).glob('model-*.safetensors'))
    if not shard_files:
        # Single file model
        shard_files = sorted(Path(MERGED_DIR).glob('*.safetensors'))
    
    for shard in shard_files:
        print(f"  Loading {shard.name} ({shard.stat().st_size / (1024**3):.1f} GB)...")
        all_tensors.update(load_file(str(shard), device='cpu'))
    
    print(f"  Total tensors loaded: {len(all_tensors)}")

    # ── Step 2: Classify and remap keys ──
    print("\nRemapping tensor keys...")
    text_tensors = OrderedDict()
    vision_keys = 0
    projector_keys = 0
    skipped_keys = []

    for key, tensor in all_tensors.items():
        if key.startswith('model.language_model.'):
            # model.language_model.embed_tokens.weight → model.embed_tokens.weight
            # model.language_model.layers.0.* → model.layers.0.*
            # model.language_model.norm.weight → model.norm.weight
            new_key = key.replace('model.language_model.', 'model.')
            text_tensors[new_key] = tensor
        elif key == 'lm_head.weight':
            text_tensors[key] = tensor
        elif key.startswith('model.vision_tower.'):
            vision_keys += 1
        elif key.startswith('model.multi_modal_projector.'):
            projector_keys += 1
        else:
            skipped_keys.append(key)

    print(f"  Text decoder tensors: {len(text_tensors)}")
    print(f"  Vision tensors (skipped): {vision_keys}")
    print(f"  Projector tensors (skipped): {projector_keys}")
    if skipped_keys:
        print(f"  Other skipped: {skipped_keys[:5]}")

    # ── Step 3: Save as sharded safetensors ──
    print(f"\nSaving text-only model to {TEXT_ONLY_DIR}/...")
    
    # Calculate total size and shard
    total_bytes = sum(t.numel() * t.element_size() for t in text_tensors.values())
    total_gb = total_bytes / (1024**3)
    print(f"  Total size: {total_gb:.1f} GB")
    
    MAX_SHARD_BYTES = 5 * 1024**3  # 5 GB per shard
    num_shards = max(1, math.ceil(total_bytes / MAX_SHARD_BYTES))
    
    if num_shards == 1:
        save_file(text_tensors, f'{TEXT_ONLY_DIR}/model.safetensors')
        # Create simple index
        weight_map = {k: 'model.safetensors' for k in text_tensors.keys()}
    else:
        # Shard the tensors
        keys = list(text_tensors.keys())
        shard_size = math.ceil(len(keys) / num_shards)
        weight_map = {}
        
        for i in range(num_shards):
            shard_keys = keys[i * shard_size : (i + 1) * shard_size]
            shard_name = f'model-{i+1:05d}-of-{num_shards:05d}.safetensors'
            shard_data = OrderedDict((k, text_tensors[k]) for k in shard_keys)
            save_file(shard_data, f'{TEXT_ONLY_DIR}/{shard_name}')
            for k in shard_keys:
                weight_map[k] = shard_name
            shard_bytes = sum(t.numel() * t.element_size() for t in shard_data.values())
            print(f"  Saved {shard_name} ({shard_bytes / (1024**3):.1f} GB, {len(shard_keys)} tensors)")
    
    # Write safetensors index
    index = {
        "metadata": {"total_size": total_bytes},
        "weight_map": weight_map,
    }
    with open(f'{TEXT_ONLY_DIR}/model.safetensors.index.json', 'w') as f:
        json.dump(index, f, indent=2)

    # ── Step 4: Build CausalLM config from VLM text_config ──
    print("\nBuilding Gemma3ForCausalLM config...")
    text_cfg = vlm_config.get('text_config', {})
    
    # Start with text_config and add required top-level fields
    causal_config = dict(text_cfg)
    causal_config['architectures'] = ['Gemma3ForCausalLM']
    causal_config['model_type'] = text_cfg.get('model_type', 'gemma3_text')
    # Ensure torch_dtype is set
    if 'torch_dtype' not in causal_config:
        causal_config['torch_dtype'] = 'bfloat16'
    # Remove any VLM-specific fields that might have leaked
    for vlm_key in ['vision_config', 'multi_modal_projector', 'image_token_index']:
        causal_config.pop(vlm_key, None)
    
    with open(f'{TEXT_ONLY_DIR}/config.json', 'w') as f:
        json.dump(causal_config, f, indent=2)
    
    print(f"  arch: {causal_config['architectures']}")
    print(f"  model_type: {causal_config.get('model_type')}")
    print(f"  hidden_size: {causal_config.get('hidden_size', '?')}")
    print(f"  num_hidden_layers: {causal_config.get('num_hidden_layers', '?')}")
    print(f"  vocab_size: {causal_config.get('vocab_size', '?')}")

    # ── Step 5: Copy tokenizer ──
    print("\nCopying tokenizer...")
    tokenizer_files = [
        'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model',
        'special_tokens_map.json', 'added_tokens.json',
    ]
    copied = 0
    for tf in tokenizer_files:
        src = Path(MERGED_DIR) / tf
        if src.exists():
            import shutil
            shutil.copy2(str(src), f'{TEXT_ONLY_DIR}/{tf}')
            copied += 1
    print(f"  Copied {copied} tokenizer files")

    # ── Summary ──
    text_size_gb = sum(
        f.stat().st_size for f in Path(TEXT_ONLY_DIR).rglob('*') if f.is_file()
    ) / (1024**3)
    safetensors_count = len(list(Path(TEXT_ONLY_DIR).glob('*.safetensors')))
    print(f"\nText-only model saved: {text_size_gb:.1f} GB ({safetensors_count} shard(s))")
    print(f"Path: {TEXT_ONLY_DIR}/")

    # Free tensor memory
    del all_tensors, text_tensors
    import gc
    gc.collect()

else:
    print("Model is already text-only. Copying to local...")
    import shutil
    if os.path.exists(TEXT_ONLY_DIR):
        shutil.rmtree(TEXT_ONLY_DIR)
    shutil.copytree(MERGED_DIR, TEXT_ONLY_DIR)
    text_size_gb = sum(
        f.stat().st_size for f in Path(TEXT_ONLY_DIR).rglob('*') if f.is_file()
    ) / (1024**3)
    print(f"Copied: {text_size_gb:.1f} GB")

## 5. Export SigLIP Vision Encoder to ONNX

SigLIP is a standard ViT that exports cleanly to ONNX.
On local machine, `trtexec` converts ONNX → TRT engine (FP16, sm_86).

**Note**: This cell loads the full VLM to access `model.model.vision_tower`.
Sub-models live on `.model`, not directly on the top-level class.

**Skip this cell if the model is text-only.**

In [ ]:
import torch
import onnx
import gc

if not IS_VLM:
    print("Text-only model — no vision encoder to export. Skipping.")
else:
    SIGLIP_ONNX = f"{ONNX_DIR}/siglip_vision.onnx"

    # Load VLM for vision extraction (text was extracted via safetensors in cell 4)
    print("Loading VLM for vision component extraction...")
    from transformers import Gemma3ForConditionalGeneration

    vlm = Gemma3ForConditionalGeneration.from_pretrained(
        MERGED_DIR,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    print("VLM loaded.")

    # Access vision tower via model.model.vision_tower (NOT vlm.vision_tower)
    print("\nExtracting SigLIP vision encoder from vlm.model.vision_tower...")
    vision_tower = vlm.model.vision_tower
    vision_tower = vision_tower.to('cuda').float().eval()  # ONNX export needs float32

    # Get image size from config
    vis_cfg = vlm_config.get('vision_config', {})
    img_size = vis_cfg.get('image_size', 384)
    vision_hidden = vis_cfg.get('hidden_size', 1152)
    patch_size = vis_cfg.get('patch_size', 14)
    num_patches = (img_size // patch_size) ** 2

    print(f"  Image size: {img_size}x{img_size}")
    print(f"  Patch size: {patch_size}")
    print(f"  Num patches: {num_patches}")
    print(f"  Hidden dim: {vision_hidden}")
    print(f"  Params: {sum(p.numel() for p in vision_tower.parameters()) / 1e6:.0f}M")

    # Create dummy input
    dummy_pixels = torch.randn(1, 3, img_size, img_size, dtype=torch.float32, device='cuda')

    # Test forward pass — vision_tower returns BaseModelOutputWithPooling
    print("\nTesting forward pass...")
    with torch.no_grad():
        test_out = vision_tower(dummy_pixels)
        # Output has .last_hidden_state attribute (confirmed from HF source)
        if hasattr(test_out, 'last_hidden_state'):
            test_tensor = test_out.last_hidden_state
            print(f"  Output type: BaseModelOutputWithPooling")
        elif isinstance(test_out, tuple):
            test_tensor = test_out[0]
            print(f"  Output type: tuple")
        else:
            test_tensor = test_out
            print(f"  Output type: {type(test_out).__name__}")
        print(f"  Output shape: {test_tensor.shape}")
        print(f"  Output dtype: {test_tensor.dtype}")

    # Wrap vision tower to return just the tensor (ONNX needs tensor output)
    class VisionWrapper(torch.nn.Module):
        def __init__(self, tower):
            super().__init__()
            self.tower = tower

        def forward(self, pixel_values):
            out = self.tower(pixel_values)
            if hasattr(out, 'last_hidden_state'):
                return out.last_hidden_state
            elif isinstance(out, tuple):
                return out[0]
            return out

    wrapper = VisionWrapper(vision_tower).eval()

    # Export to ONNX
    print(f"\nExporting to ONNX: {SIGLIP_ONNX}")
    torch.onnx.export(
        wrapper,
        dummy_pixels,
        SIGLIP_ONNX,
        input_names=['pixel_values'],
        output_names=['image_features'],
        dynamic_axes={
            'pixel_values': {0: 'batch'},
            'image_features': {0: 'batch'},
        },
        opset_version=17,
        do_constant_folding=True,
    )

    # Verify ONNX
    onnx_model = onnx.load(SIGLIP_ONNX)
    onnx.checker.check_model(onnx_model, full_check=True)
    siglip_size_mb = os.path.getsize(SIGLIP_ONNX) / (1024**2)
    print(f"SigLIP ONNX exported: {siglip_size_mb:.0f} MB")
    print(f"ONNX check: PASSED")

    del wrapper, vision_tower, test_out, test_tensor, dummy_pixels, onnx_model
    gc.collect()
    torch.cuda.empty_cache()
    print("Vision encoder export complete.")

## 6. Export Projector to ONNX

`Gemma3MultiModalProjector` bridges SigLIP output to Gemma3 text input:
- `mm_input_projection_weight` (nn.Parameter): vision_hidden → text_hidden
- `mm_soft_emb_norm` (RMSNorm): normalization
- `avg_pool` (AvgPool2d): spatial downsampling

Accessed via `vlm.model.multi_modal_projector` (NOT `vlm.multi_modal_projector`).

**Skip this cell if the model is text-only.**

In [ ]:
import torch
import onnx
import gc

if not IS_VLM:
    print("Text-only model — no projector to export. Skipping.")
else:
    PROJECTOR_ONNX = f"{ONNX_DIR}/gemma_projector.onnx"

    # Access projector via vlm.model.multi_modal_projector (NOT vlm.multi_modal_projector)
    print("Extracting multi-modal projector from vlm.model.multi_modal_projector...")
    projector = vlm.model.multi_modal_projector
    projector = projector.to('cuda').float().eval()  # ONNX export needs float32

    vis_cfg = vlm_config.get('vision_config', {})
    vision_hidden = vis_cfg.get('hidden_size', 1152)
    img_size = vis_cfg.get('image_size', 384)
    patch_size = vis_cfg.get('patch_size', 14)
    num_patches = (img_size // patch_size) ** 2

    print(f"  Input: ({num_patches}, {vision_hidden}) per image")
    print(f"  Params: {sum(p.numel() for p in projector.parameters()) / 1e6:.1f}M")
    # Show projector structure
    print(f"  Attributes: {[n for n, _ in projector.named_children()]}")
    print(f"  Parameters: {[n for n, _ in projector.named_parameters()]}")

    # Dummy input: batch of vision features (output of SigLIP last_hidden_state)
    dummy_features = torch.randn(
        1, num_patches, vision_hidden,
        dtype=torch.float32, device='cuda'
    )

    # Test forward pass
    print("\nTesting forward pass...")
    with torch.no_grad():
        proj_out = projector(dummy_features)
        if isinstance(proj_out, tuple):
            proj_out = proj_out[0]
        print(f"  Output shape: {proj_out.shape}")
        print(f"  Output dtype: {proj_out.dtype}")

    # Export to ONNX
    print(f"\nExporting to ONNX: {PROJECTOR_ONNX}")
    torch.onnx.export(
        projector,
        dummy_features,
        PROJECTOR_ONNX,
        input_names=['image_features'],
        output_names=['projected_tokens'],
        dynamic_axes={
            'image_features': {0: 'batch'},
            'projected_tokens': {0: 'batch'},
        },
        opset_version=17,
        do_constant_folding=True,
    )

    # Verify
    onnx_model = onnx.load(PROJECTOR_ONNX)
    onnx.checker.check_model(onnx_model, full_check=True)
    proj_size_mb = os.path.getsize(PROJECTOR_ONNX) / (1024**2)
    print(f"Projector ONNX exported: {proj_size_mb:.1f} MB")
    print(f"ONNX check: PASSED")

    # Free VLM memory — no longer needed
    del projector, proj_out, dummy_features, vlm, onnx_model
    gc.collect()
    torch.cuda.empty_cache()
    print("\nVLM unloaded. GPU memory freed.")

## 7. Verify Text-Only Model Loads Standalone

Quick sanity check: load the extracted text-only model as `Gemma3ForCausalLM`
to confirm TRT-LLM's `convert_checkpoint.py` will accept it.

In [ ]:
import torch
import json
from pathlib import Path

# Check saved config
with open(f'{TEXT_ONLY_DIR}/config.json') as f:
    text_config = json.load(f)

arch = text_config.get('architectures', ['unknown'])[0]
print(f"Saved architecture: {arch}")
print(f"model_type: {text_config.get('model_type', '?')}")
print(f"hidden_size: {text_config.get('hidden_size', '?')}")
print(f"num_hidden_layers: {text_config.get('num_hidden_layers', '?')}")
print(f"vocab_size: {text_config.get('vocab_size', '?')}")

# Verify no vision keys leaked into the text model
print("\nChecking for leaked vision keys...")
from safetensors import safe_open
leaked = []
for sf in sorted(Path(TEXT_ONLY_DIR).glob('*.safetensors')):
    with safe_open(str(sf), framework='pt') as f:
        for key in f.keys():
            if 'vision' in key.lower() or 'projector' in key.lower():
                leaked.append(key)

if leaked:
    print(f"  WARNING: {len(leaked)} vision keys found in text model:")
    for k in leaked[:5]:
        print(f"    {k}")
    print("  These will cause convert_checkpoint.py to fail.")
    print("  Manual tensor filtering needed.")
else:
    print("  No vision keys found. Clean text-only model.")

# Quick load test (just config, not full weights — saves time)
from transformers import AutoConfig
try:
    test_config = AutoConfig.from_pretrained(TEXT_ONLY_DIR)
    print(f"\nAutoConfig loads: {type(test_config).__name__}")
    print(f"Config valid: YES")
except Exception as e:
    print(f"\nAutoConfig failed: {e}")
    print("May need manual config fixup.")

# File listing
print(f"\nText-only model contents:")
total_gb = 0
for f in sorted(Path(TEXT_ONLY_DIR).iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / (1024**2)
        total_gb += size_mb / 1024
        print(f"  {f.name:45s} {size_mb:>8.1f} MB")
print(f"  {'Total':45s} {total_gb:>7.1f} GB")

## 8. Save All Artifacts to Google Drive

Saves:
- Text-only model (~21 GB) → for `convert_checkpoint.py` + `trtllm-build`
- SigLIP ONNX (~1.5 GB) → for `trtexec`
- Projector ONNX (~50 MB) → for `trtexec`
- Manifest JSON with metadata

In [ ]:
import os
import shutil
import json
from pathlib import Path
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ARTIFACTS = '/content/drive/MyDrive/gemma3-12b-legal-trt-artifacts'

print(f"Saving artifacts to: {DRIVE_ARTIFACTS}")
print(f"This takes 10-20 minutes for ~22 GB...\n")

# Create directory structure
os.makedirs(f'{DRIVE_ARTIFACTS}/text_only_model', exist_ok=True)
os.makedirs(f'{DRIVE_ARTIFACTS}/onnx', exist_ok=True)

# ── Copy text-only model ──
print("Copying text-only model...")
text_dest = f'{DRIVE_ARTIFACTS}/text_only_model'
if os.path.exists(text_dest):
    shutil.rmtree(text_dest)
shutil.copytree(TEXT_ONLY_DIR, text_dest, dirs_exist_ok=True)
text_gb = sum(f.stat().st_size for f in Path(text_dest).rglob('*') if f.is_file()) / (1024**3)
print(f"  Text model: {text_gb:.1f} GB")

# ── Copy ONNX files ──
onnx_files = {}
siglip_src = f'{ONNX_DIR}/siglip_vision.onnx'
proj_src = f'{ONNX_DIR}/gemma_projector.onnx'

if os.path.exists(siglip_src):
    print("Copying SigLIP ONNX...")
    shutil.copy2(siglip_src, f'{DRIVE_ARTIFACTS}/onnx/siglip_vision.onnx')
    siglip_mb = os.path.getsize(siglip_src) / (1024**2)
    print(f"  SigLIP: {siglip_mb:.0f} MB")
    onnx_files['siglip_vision.onnx'] = siglip_mb

if os.path.exists(proj_src):
    print("Copying Projector ONNX...")
    shutil.copy2(proj_src, f'{DRIVE_ARTIFACTS}/onnx/gemma_projector.onnx')
    proj_mb = os.path.getsize(proj_src) / (1024**2)
    print(f"  Projector: {proj_mb:.1f} MB")
    onnx_files['gemma_projector.onnx'] = proj_mb

# ── Read back text config for manifest ──
with open(f'{TEXT_ONLY_DIR}/config.json') as f:
    saved_text_config = json.load(f)

# ── Create manifest ──
manifest = {
    'created': datetime.now().isoformat(),
    'source_model': MERGED_DIR,
    'source_arch': vlm_config.get('architectures', ['unknown'])[0],
    'is_vlm': IS_VLM,
    'text_model': {
        'path': 'text_only_model/',
        'arch': 'Gemma3ForCausalLM',
        'size_gb': round(text_gb, 1),
        'dtype': 'bfloat16',
        'hidden_size': saved_text_config.get('hidden_size', 3840),
        'num_layers': saved_text_config.get('num_hidden_layers', 48),
        'vocab_size': saved_text_config.get('vocab_size', 262144),
    },
    'onnx_models': onnx_files,
    'next_steps': [
        'Download text_only_model/ to local machine',
        'Download onnx/ files to local machine',
        'In Docker (TRT-LLM v0.21.0): python3 examples/gemma/convert_checkpoint.py --ckpt-type hf --model-dir /models/text_only --use-weight-only-with-precision int4 --dtype bfloat16 --world-size 1 --output-model-dir /models/int4_checkpoint',
        'In Docker: trtllm-build --checkpoint_dir /models/int4_checkpoint --gemm_plugin auto --gpt_attention_plugin auto --max_batch_size 4 --max_input_len 2048 --max_seq_len 4096 --output_dir /models/engine_int4',
        'In Docker: trtexec --onnx=/models/siglip_vision.onnx --saveEngine=/models/siglip_vision.engine --fp16',
        'In Docker: trtexec --onnx=/models/gemma_projector.onnx --saveEngine=/models/gemma_projector.engine --fp16',
    ],
}

with open(f'{DRIVE_ARTIFACTS}/export_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# ── Summary ──
total_gb = sum(
    f.stat().st_size for f in Path(DRIVE_ARTIFACTS).rglob('*') if f.is_file()
) / (1024**3)
print(f"\n{'='*60}")
print(f"All artifacts saved to Google Drive: {total_gb:.1f} GB")
print(f"Path: {DRIVE_ARTIFACTS}")
print(f"{'='*60}")
print(f"\nDownload from: https://drive.google.com/")
print(f"Folder: My Drive / gemma3-12b-legal-trt-artifacts/")

## 9. Next Steps (On Local Machine)

Download the artifacts from Google Drive (~22 GB total), then build engines locally.

### Download
```
From: My Drive / gemma3-12b-legal-trt-artifacts/
To:   c:\Users\james\Videos\deeds-web-app\trt_artifacts\
```

### Build Engines in Docker (RTX 3060 Ti)
```bash
# 1. Build Docker image
docker build -f Dockerfile.trtllm -t legal-ai-trtllm .

# 2. Start container with GPU + model mounts
docker run --gpus all -it \
  -v ./trt_artifacts/text_only_model:/models/text_only:ro \
  -v ./trt_artifacts/onnx:/models/onnx:ro \
  -v ./engines:/models/engines \
  legal-ai-trtllm bash

# 3. Inside container — convert text model to INT4 checkpoint
#    Note: Gemma convert_checkpoint.py uses dashes, not underscores
cd /workspace/tensorrt-llm
python3 examples/gemma/convert_checkpoint.py \
  --ckpt-type hf \
  --model-dir /models/text_only \
  --use-weight-only-with-precision int4 \
  --dtype bfloat16 \
  --world-size 1 \
  --output-model-dir /models/int4_checkpoint

# 4. Build TRT engine (compiles for sm_86 / RTX 3060 Ti)
trtllm-build \
  --checkpoint_dir /models/int4_checkpoint \
  --gemm_plugin auto \
  --gpt_attention_plugin auto \
  --max_batch_size 4 \
  --max_input_len 2048 \
  --max_seq_len 4096 \
  --output_dir /models/engines/gemma3_12b_int4

# 5. Build SigLIP engine (check image_size from config — may be 384 or 896)
#    Use the image_size from Cell 3 output
trtexec \
  --onnx=/models/onnx/siglip_vision.onnx \
  --saveEngine=/models/engines/siglip_vision.engine \
  --fp16 \
  --optShapes=pixel_values:1x3x384x384 \
  --maxShapes=pixel_values:4x3x384x384

# 6. Build Projector engine
trtexec \
  --onnx=/models/onnx/gemma_projector.onnx \
  --saveEngine=/models/engines/gemma_projector.engine \
  --fp16
```

### Deploy via Triton
```bash
# Restore Triton configs from archive
cp -r deeds_labs/legacy-projects/triton_models/ triton_models/

# Start Triton
docker compose -f docker-compose.triton.yml up triton-legal-ai -d
curl http://localhost:8099/v2/health/ready
```

### Test
```bash
# Text-only
curl -X POST http://localhost:5173/api/ai/tensorrt \
  -H "Content-Type: application/json" \
  -d '{"prompt": "Analyze breach of contract liability"}'

# VLM (image + text)
curl -X POST http://localhost:5173/api/ai/tensorrt/vlm \
  -F "image=@evidence_photo.jpg" \
  -F "prompt=Describe this evidence document"
```

See `next_steps/TRT_ENGINE_BUILD_STEPS.md` for full details.